# 21 · Spectral shape analysis with the Laplace–Beltrami operator

The spectrum of the Laplace–Beltrami operator is an **intrinsic** shape
fingerprint ("Shape-DNA"): invariant to rotation, translation, and isometric
bending. Here we build surfaces from omnibias **charts**, assemble a discrete
LBO, and use its spectrum and heat-kernel signature for classification and
segmentation. The continuous operator these discretize is exactly the
`laplace_beltrami` of `omnibias-geometry`.

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.cluster.vq import kmeans2

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

rng = np.random.default_rng(0)

## 1. Surfaces from charts + a cotangent Laplace–Beltrami operator

`grid_surface` triangulates a parametric chart `φ(u, v)`; `cot_laplacian` builds
the standard cotangent stiffness `L` and lumped mass `M` (vertex areas — the
discrete `√|g|`).

In [ ]:
def grid_surface(phi, nu=30, nv=30, u_range=(0.15, np.pi - 0.15)):
    us = np.linspace(*u_range, nu)
    vs = np.linspace(0, 2 * np.pi, nv, endpoint=False)
    V = np.array([phi(u, v) for u in us for v in vs], dtype=float)
    idx = lambda i, j: i * nv + (j % nv)
    F = []
    for i in range(nu - 1):
        for j in range(nv):
            a, b = idx(i, j), idx(i, j + 1)
            c, d = idx(i + 1, j), idx(i + 1, j + 1)
            F.append((a, b, d)); F.append((a, d, c))
    return V, np.array(F)

def cot_laplacian(V, F):
    n = len(V)
    W = np.zeros((n, n)); mass = np.zeros(n)
    for i, j, k in F:
        vi, vj, vk = V[i], V[j], V[k]
        area = 0.5 * np.linalg.norm(np.cross(vj - vi, vk - vi)) + 1e-12
        for a in (i, j, k):
            mass[a] += area / 3.0
        def cot(p, q, r):
            u, w = p - r, q - r
            return np.dot(u, w) / (np.linalg.norm(np.cross(u, w)) + 1e-12)
        for (p, q, r) in [(i, j, k), (j, k, i), (k, i, j)]:
            c = 0.5 * cot(V[p], V[q], V[r])
            W[p, q] += c; W[q, p] += c
    L = np.diag(W.sum(1)) - W
    return L, mass

def lbo_spectrum(phi, k=16, **kw):
    V, F = grid_surface(phi, **kw)
    L, mass = cot_laplacian(V, F)
    vals, vecs = eigsh(sp.csr_matrix(L), k=k, M=sp.diags(mass), sigma=1e-8, which="LM")
    order = np.argsort(vals.real)
    return vals.real[order], vecs[:, order], V, F

shapes = {
    "sphere":    lambda u, v: (np.sin(u) * np.cos(v), np.sin(u) * np.sin(v), np.cos(u)),
    "ellipsoid": lambda u, v: (1.6 * np.sin(u) * np.cos(v), np.sin(u) * np.sin(v), np.cos(u)),
    "torus":     lambda u, v: ((1.0 + 0.4 * np.cos(2 * u)) * np.cos(v),
                               (1.0 + 0.4 * np.cos(2 * u)) * np.sin(v), 0.4 * np.sin(2 * u)),
}

## 2. Shape-DNA: the LBO spectrum is an intrinsic fingerprint

On the unit sphere the eigenvalues approach the analytic `l(l+1) = 0, 2, 6, 12,
…` (with multiplicities `2l+1`). Different shapes have visibly different spectra.

In [ ]:
spectra, eigvecs, meshes = {}, {}, {}
for name, phi in shapes.items():
    vals, vecs, V, F = lbo_spectrum(phi, k=16)
    spectra[name] = vals; eigvecs[name] = (vals, vecs); meshes[name] = (V, F)

print("sphere eigenvalues:", np.round(spectra["sphere"][:9], 2))
print("analytic l(l+1)  :", [0, 2, 2, 2, 6, 6, 6, 6, 6])

fig, ax = plt.subplots(figsize=(7.4, 3.8))
for (name, vals), col in zip(spectra.items(), [PRIMARY, ACCENT, GOOD]):
    ax.plot(range(1, 16), vals[1:16], "o-", color=col, label=name, ms=4)
ax.set_xlabel("index"); ax.set_ylabel("eigenvalue λ"); ax.set_title("Shape-DNA spectra")
ax.legend()
plt.tight_layout()

## 3. Intrinsic classification by spectral distance

The spectrum is rotation/translation invariant. We rotate each shape randomly,
recompute its Shape-DNA, and classify by nearest spectrum — recovering the true
class despite the random pose.

In [ ]:
def rand_rot():
    A = rng.standard_normal((3, 3)); Q, _ = np.linalg.qr(A)
    return Q * np.sign(np.linalg.det(Q))

names = list(shapes)
queries = []
for name in names:
    Q = rand_rot()
    phi = shapes[name]
    rphi = (lambda u, v, p=phi, Q=Q: tuple(Q @ np.array(p(u, v))))
    vals, _, _, _ = lbo_spectrum(rphi, k=16)
    queries.append((name, vals))

ref = {n: spectra[n][1:] for n in names}   # drop trivial 0 eigenvalue
correct = 0
for true_name, q in queries:
    dists = {n: np.linalg.norm(q[1:] - ref[n]) for n in names}
    pred = min(dists, key=dists.get)
    correct += (pred == true_name)
    print(f"query {true_name:>9} (random pose) -> predicted {pred:>9}")
print(f"\nintrinsic classification accuracy: {correct}/{len(queries)}")

## 4. Spectral segmentation via the Heat-Kernel Signature

The HKS `k_t(x) = Σ_i e^{−λ_i t} φ_i(x)²` is a multi-scale, intrinsic per-point
descriptor. Clustering vertices by their HKS partitions the surface into
intrinsically-similar regions.

In [ ]:
vals, vecs, V, F = lbo_spectrum(shapes["torus"], k=24)
ts = np.geomspace(0.02, 1.0, 8)
lam, phi2 = vals[1:], vecs[:, 1:] ** 2          # drop trivial mode; (k-1,), (n, k-1)
hks = phi2 @ np.exp(-np.outer(lam, ts))         # (n, T): k_t(x) = Σ_i e^{-λ_i t} φ_i(x)²
feat = (hks - hks.mean(0)) / (hks.std(0) + 1e-9)
_, labels = kmeans2(feat, 4, rng=np.random.default_rng(0), minit="++")

fig = plt.figure(figsize=(5.6, 4.6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(V[:, 0], V[:, 1], V[:, 2], c=labels, cmap="tab10", s=10)
ax.set_title("HKS spectral segmentation (torus)")
plt.tight_layout()

## Takeaway

Shape-DNA, HKS, intrinsic classification and segmentation all derive from the
LBO spectrum — the discrete face of omnibias's exact `laplace_beltrami`. With the
pullback metric, the same pipeline runs directly off a (possibly learned) chart.